# Loan Approval Prediction - Model Development

**Dataset**: Give Me Some Credit (Kaggle)

**Goal**: Predict serious financial distress in next 2 years

**Sections**:
1. Data Loading & Exploration
2. Data Preprocessing
3. Feature Engineering
4. Model Training & Comparison
5. Model Selection with MLflow
6. Final Model Evaluation

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ML libraries
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Imbalanced data handling
from imblearn.over_sampling import SMOTE

# MLflow
import mlflow
import mlflow.sklearn
import mlflow.xgboost

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# MLflow setup — falls back to local tracking if server is not running
try:
    mlflow.set_tracking_uri("http://localhost:5000")
    mlflow.set_experiment("loan-approval-prediction-notebook")
    print("MLflow connected to http://localhost:5000")
except Exception:
    mlflow.set_tracking_uri("mlruns")
    mlflow.set_experiment("loan-approval-prediction-notebook")
    print("MLflow server not running — using local ./mlruns directory")

print("All libraries imported successfully!")

## 1️⃣ Data Loading & Exploration

In [ ]:
# Load data
df = pd.read_csv('../data/raw/cs-training.csv', index_col=0)

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
df.head()

In [ ]:
# Basic info
print("Dataset Information:")
print(df.info())
print("\n" + "="*50)
print("\nStatistical Summary:")
print(df.describe())

In [ ]:
# Check missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Percentage': missing_pct
}).sort_values('Missing Count', ascending=False)

print("Missing Values:")
print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
# Target variable distribution
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
target_counts = df['SeriousDlqin2yrs'].value_counts()
ax[0].bar(['No Default (0)', 'Default (1)'], target_counts.values, color=['green', 'red'])
ax[0].set_title('Target Distribution', fontsize=14, fontweight='bold')
ax[0].set_ylabel('Count')

# Percentage
target_pct = df['SeriousDlqin2yrs'].value_counts(normalize=True) * 100
ax[1].pie(target_pct, labels=['No Default (0)', 'Default (1)'], 
          autopct='%1.2f%%', colors=['green', 'red'], startangle=90)
ax[1].set_title('Target Percentage', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"Class Imbalance Ratio: {target_pct.values[0] / target_pct.values[1]:.2f}:1")

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 10))
correlation = df.corr(numeric_only=True)
sns.heatmap(correlation, annot=True, cmap='coolwarm', center=0, 
            fmt='.2f', linewidths=0.5)
plt.title('Feature Correlation Matrix', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 2️⃣ Data Preprocessing

In [ ]:
# Create a copy
df_clean = df.copy()

# Handle MonthlyIncome missing values
print(f"MonthlyIncome missing: {df_clean['MonthlyIncome'].isnull().sum()}")
df_clean['MonthlyIncome'] = df_clean['MonthlyIncome'].fillna(df_clean['MonthlyIncome'].median())

# Handle NumberOfDependents missing values
print(f"NumberOfDependents missing: {df_clean['NumberOfDependents'].isnull().sum()}")
df_clean['NumberOfDependents'] = df_clean['NumberOfDependents'].fillna(df_clean['NumberOfDependents'].median())

# Handle age outliers (age = 0)
print(f"\nAge = 0 count: {(df_clean['age'] == 0).sum()}")
df_clean['age'] = df_clean['age'].replace(0, df_clean['age'].median())

# Handle extreme outliers in RevolvingUtilizationOfUnsecuredLines
Q1 = df_clean['RevolvingUtilizationOfUnsecuredLines'].quantile(0.25)
Q3 = df_clean['RevolvingUtilizationOfUnsecuredLines'].quantile(0.75)
IQR = Q3 - Q1
upper_limit = Q3 + 3 * IQR
df_clean.loc[df_clean['RevolvingUtilizationOfUnsecuredLines'] > upper_limit, 
             'RevolvingUtilizationOfUnsecuredLines'] = upper_limit

print(f"\nData cleaned. Shape: {df_clean.shape}")
print(f"Remaining missing values: {df_clean.isnull().sum().sum()}")

## 3️⃣ Feature Engineering

In [ ]:
# Create new features
df_features = df_clean.copy()

# 1. Debt to Income Ratio
df_features['DebtToIncomeRatio'] = df_features['DebtRatio'] / (df_features['MonthlyIncome'] + 1)

# 2. Credit Utilization Category (use -np.inf to include 0)
df_features['CreditUtilization_Category'] = pd.cut(
    df_features['RevolvingUtilizationOfUnsecuredLines'],
    bins=[-np.inf, 0.3, 0.6, 1.0, float('inf')],
    labels=[0, 1, 2, 3]
).astype(int)

# 3. Age Group (use -np.inf and inf for safe boundaries)
df_features['AgeGroup'] = pd.cut(
    df_features['age'],
    bins=[-np.inf, 30, 45, 60, float('inf')],
    labels=[0, 1, 2, 3]
).astype(int)

# 4. Total Past Due Events
df_features['TotalPastDue'] = (
    df_features['NumberOfTime30-59DaysPastDueNotWorse'] +
    df_features['NumberOfTime60-89DaysPastDueNotWorse'] +
    df_features['NumberOfTimes90DaysLate']
)

# 5. Has Past Due
df_features['HasPastDue'] = (df_features['TotalPastDue'] > 0).astype(int)

# 6. Income per Dependent
df_features['IncomePerDependent'] = df_features['MonthlyIncome'] / (df_features['NumberOfDependents'] + 1)

# 7. Log Monthly Income (handles skewness)
df_features['LogMonthlyIncome'] = np.log1p(df_features['MonthlyIncome'])

# 8. Loans per Credit Line
df_features['LoansPerCreditLine'] = df_features['NumberRealEstateLoansOrLines'] / (df_features['NumberOfOpenCreditLinesAndLoans'] + 1)

print(f"Features created. New shape: {df_features.shape}")
print(f"\nNew features: {[col for col in df_features.columns if col not in df_clean.columns]}")

## 4️⃣ Model Training & Comparison

In [ ]:
# Prepare data
X = df_features.drop('SeriousDlqin2yrs', axis=1)
y = df_features['SeriousDlqin2yrs']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"\nClass distribution in training set:")
print(y_train.value_counts(normalize=True))

In [ ]:
# Handle class imbalance with SMOTE
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"Original training size: {X_train.shape}")
print(f"Balanced training size: {X_train_balanced.shape}")
print(f"\nClass distribution after SMOTE:")
print(y_train_balanced.value_counts(normalize=True))

In [ ]:
# Function to evaluate model
def evaluate_model(model, X_train, y_train, X_test, y_test, model_name):
    """
    Train and evaluate a model with MLflow tracking
    """
    try:
        ctx = mlflow.start_run(run_name=model_name)
        ctx.__enter__()
        use_mlflow = True
    except Exception:
        use_mlflow = False

    # Train model
    model.fit(X_train, y_train)

    # Predictions
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc_roc = roc_auc_score(y_test, y_pred_proba)

    # Log to MLflow if available
    if use_mlflow:
        try:
            mlflow.log_param("model_type", model_name)
            mlflow.log_param("train_samples", len(X_train))
            mlflow.log_param("test_samples", len(X_test))
            mlflow.log_metric("accuracy", accuracy)
            mlflow.log_metric("precision", precision)
            mlflow.log_metric("recall", recall)
            mlflow.log_metric("f1_score", f1)
            mlflow.log_metric("auc_roc", auc_roc)
            if 'XGB' in model_name:
                mlflow.xgboost.log_model(model, "model")
            else:
                mlflow.sklearn.log_model(model, "model")
        except Exception as e:
            print(f"  (MLflow logging skipped: {e})")
        finally:
            ctx.__exit__(None, None, None)

    # Print results
    print(f"\n{'='*60}")
    print(f"Model: {model_name}")
    print(f"{'='*60}")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print(f"AUC-ROC:   {auc_roc:.4f}")

    return {
        'model': model,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'auc_roc': auc_roc,
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba
    }

print("Evaluation function defined")

### Train Multiple Models

In [ ]:
# Initialize models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        min_samples_split=10,
        random_state=42,
        n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        min_child_weight=1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
}

# Train and evaluate all models
results = {}
for name, model in models.items():
    results[name] = evaluate_model(
        model, X_train_balanced, y_train_balanced, X_test, y_test, name
    )

In [ ]:
# Compare models
comparison_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [results[m]['accuracy'] for m in results],
    'Precision': [results[m]['precision'] for m in results],
    'Recall': [results[m]['recall'] for m in results],
    'F1 Score': [results[m]['f1_score'] for m in results],
    'AUC-ROC': [results[m]['auc_roc'] for m in results]
}).sort_values('AUC-ROC', ascending=False)

print("\n📊 Model Comparison:")
print(comparison_df.to_string(index=False))

# Visualize comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

metrics = ['Accuracy', 'Precision', 'Recall', 'AUC-ROC']
for idx, metric in enumerate(metrics):
    ax = axes[idx // 2, idx % 2]
    comparison_df.plot(x='Model', y=metric, kind='bar', ax=ax, legend=False)
    ax.set_title(f'{metric} Comparison', fontsize=14, fontweight='bold')
    ax.set_ylabel(metric)
    ax.set_xlabel('')
    ax.set_ylim(0, 1)
    
    # Add value labels on bars
    for container in ax.containers:
        ax.bar_label(container, fmt='%.3f')

plt.tight_layout()
plt.show()

## 5️⃣ Best Model Selection

In [ ]:
# Select best model based on AUC-ROC
best_model_name = comparison_df.iloc[0]['Model']
best_model_results = results[best_model_name]

print(f"🏆 Best Model: {best_model_name}")
print(f"AUC-ROC: {best_model_results['auc_roc']:.4f}")

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, best_model_results['y_pred'])

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No Default', 'Default'],
            yticklabels=['No Default', 'Default'])
plt.title(f'Confusion Matrix - {best_model_name}', fontsize=14, fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

print("\nClassification Report:")
print(classification_report(y_test, best_model_results['y_pred'], 
                          target_names=['No Default', 'Default']))

In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, best_model_results['y_pred_proba'])

plt.figure(figsize=(10, 6))
plt.plot(fpr, tpr, linewidth=2, label=f'{best_model_name} (AUC = {best_model_results["auc_roc"]:.4f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Feature Importance (for tree-based models)
if best_model_name in ['Random Forest', 'XGBoost', 'LightGBM']:
    feature_importance = pd.DataFrame({
        'feature': X_train.columns,
        'importance': best_model_results['model'].feature_importances_
    }).sort_values('importance', ascending=False).head(15)
    
    plt.figure(figsize=(10, 8))
    plt.barh(feature_importance['feature'], feature_importance['importance'])
    plt.xlabel('Importance', fontsize=12)
    plt.title(f'Top 15 Feature Importances - {best_model_name}', fontsize=14, fontweight='bold')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

## 6️⃣ Save Best Model

In [ ]:
# Save model + metrics
import joblib
import json
import os

# Determine output paths based on environment (local vs Colab)
if os.path.exists('../models') or os.path.exists('../backend'):
    # Running from notebooks/ folder locally
    model_dir = '../models'
    metrics_path = '../metrics.json'
else:
    # Running in Colab or from project root
    model_dir = 'models'
    metrics_path = 'metrics.json'

os.makedirs(model_dir, exist_ok=True)

# Save model
model_path = os.path.join(model_dir, 'model.pkl')
joblib.dump(best_model_results['model'], model_path)
print(f"Model saved to: {model_path}")
print(f"Model type: {type(best_model_results['model']).__name__}")

# Save metrics.json (used by the FastAPI /model_info and /metrics endpoints)
cm = confusion_matrix(y_test, best_model_results['y_pred'])
metrics = {
    "roc_auc": round(best_model_results['auc_roc'], 4),
    "precision": round(best_model_results['precision'], 4),
    "recall": round(best_model_results['recall'], 4),
    "f1_score": round(best_model_results['f1_score'], 4),
    "accuracy": round(best_model_results['accuracy'], 4),
    "tn": int(cm[0, 0]),
    "fp": int(cm[0, 1]),
    "fn": int(cm[1, 0]),
    "tp": int(cm[1, 1]),
    "model_type": best_model_name,
    "train_samples": len(X_train_balanced),
    "test_samples": len(X_test)
}

with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f"Metrics saved to: {metrics_path}")

print(f"\nThis model is now ready for the FastAPI backend.")
print(f"Download model.pkl and metrics.json, then run start_services.bat to launch the app.")

## Summary

**Key Findings**:
1. Dataset has 6.7% default rate (class imbalance)
2. Used SMOTE to balance training data
3. Created 8 new features for better predictions
4. Compared 4 models: Logistic Regression, Random Forest, XGBoost, LightGBM
5. Best model achieves ~86% AUC-ROC

**Features created**:
DebtToIncomeRatio, CreditUtilization_Category, AgeGroup, TotalPastDue, HasPastDue, IncomePerDependent, LogMonthlyIncome, LoansPerCreditLine

**Next Steps**:
- The saved `models/model.pkl` is used directly by the FastAPI backend
- Run `start_services.bat` to launch both the API and Streamlit frontend
- Or deploy with `docker-compose up --build`